# Test Stage 10 — The Feedback Atlas, and the Operating-Point Designer

This stage reproduces the flagship measurement of Herkenhoff *et al.* (their Fig. 8): a
scan over every feedback **gain × phase** setting, mapping what each one does to the
resonator — its quality factor Q, its centre frequency, and its **effective temperature**,
a map the paper derives but never draws. Settings that drive the resonator into
self-oscillation are found automatically and marked black.

Then it goes one step further: the measured maps become a **designer**. Rather than
reading numbers off a heatmap, you ask for what you want — "give me Q = 1000", "shift the
resonance 150 kHz" — and get back the settings, the costs (temperature, ring-up time,
stability margin), and a measurement plan.

## The physics, in three sentences

Feedback in phase with the resonator's losses either replaces or steals energy, so Q
rises or falls — and with it the apparent temperature, in strict proportion:
`T_eff/T0 = Q_eff/Q0` (the fluctuation-dissipation theorem, the paper's Eq. 8). Feedback a
quarter turn away changes the oscillation rate instead, moving the centre frequency, which
is how the ion is decoupled. Everything between is a mixture, and past
`gain × cos(phase) = 1` the losses go negative and the resonator oscillates on its own —
useful on purpose (Step 5), destructive by accident (the black regions).

## What you need

| | |
|---|---|
| Stages 1–9 | Passing |
| Bitstream | The Stage 9 build (noise sources present — Step 0 checks) |
| Time | ~10 minutes at the default grid; the grid is a knob |
| Cables | None. Set `EMULATION = False` to run the identical scan on the real coil |

---

### Simulation or the real trap?

**`EMULATION = False` is not a switch — read this before believing it.** That flag currently
gates one assertion and nothing else. Moving to the real coil is a small piece of work, not a
setting.

Nothing needs rebuilding. The resonator is disabled at power-on, so leaving it disabled is all
the *hardware* requires. What changes is in this notebook:

| | Emulated coil | Real coil |
|---|---|---|
| Where the resonance is, and how wide | `reso.config(F0, Q_SET)` sets both, and `LW0` follows | Neither is known. **Measure them first**, then feed those numbers in wherever `reso.f0_q` and `LW0` appear |
| Is this point self-oscillating? | Switch the coil's noise off and see whether the signal survives | You cannot switch off a real coil's thermal noise. Use the coherence measure instead: an oscillation is a coherent tone, a thermal bump is not |
| Where does the loop stop being linear? | Drop the coil's noise 12 dB and check the bump area falls 16× | You cannot change a real coil's thermal drive. Inject a known tone at two amplitudes and check the response scales instead |
| The scale of everything | Q ≈ 180, linewidth ≈ 222 kHz | A cryogenic coil is Q ~ 10³–10⁴, so the linewidth is 8–20 kHz. Sweep spans, averaging choices and settle times all need re-tuning |

The checks that survive intact are the ones that need no known answer: the temperature law
(Step 3), narrowing versus broadening agreement, reproducibility on cold re-measurement, and
the atlas predicting a self-oscillation threshold that bisection then walks to. Those are the
ones worth trusting on real data.


## Setup — drivers and capture (identical to Stage 9, proven there)

> **Real coil.** The driver cell is unchanged. The capture cell is unchanged. Step 0 is where
> the emulation begins: it configures the resonator, enables it, and turns on its noise. For
> real work, leave all of that out — the board then listens to the actual coil.


In [ ]:
import os, mmap, struct, time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

FS_CLK = 125e6
F0     = 40.0e6          # the coil being emulated: ~40 MHz
Q_SET  = 180.0           # ~180 at room temperature
SETTLE = 0.02
N_AVG  = 32

# =============================================================================
#  Environment shims. This board carries an older numpy and matplotlib than a
#  current desktop, and two conveniences used below arrived later than what is
#  installed here: nan_to_num's `nan=` keyword needs numpy >= 1.17, and
#  Axes.set_facecolor needs matplotlib >= 2.0. Provided once, rather than as
#  version tests scattered through the analysis cells.
# =============================================================================
def nan_fill(a, value):
    """np.nan_to_num(a, nan=value), for numpy versions without the keyword."""
    out = np.array(a, dtype=float, copy=True)
    out[~np.isfinite(out)] = value
    return out


def set_bg(ax, colour):
    """Axes background colour, across the matplotlib 1.x -> 2.x rename."""
    if hasattr(ax, 'set_facecolor'):
        ax.set_facecolor(colour)
    else:
        ax.set_axis_bgcolor(colour)


# =============================================================================
#  Drivers - the Stage 8 set, with the resonator gaining its two noise knobs.
#  CRITICAL RULE throughout: one 32-bit register access at a time.
# =============================================================================
class Bus(object):
    def __init__(self, base, span=0x1000):
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m  = mmap.mmap(self.fd, span, offset=base)
    def w(self, off, val):
        self.m[off:off + 4] = struct.pack('<I', val & 0xFFFFFFFF)
    def r(self, off):
        return struct.unpack('<I', self.m[off:off + 4])[0]

def s32(v):
    return v - (1 << 32) if v & 0x80000000 else v


class Acq(Bus):
    """Acquisition system as narrowband voltmeter (PEEK at DC) and as
    recorder (capture to memory - defined in the next cell)."""
    def __init__(self):
        Bus.__init__(self, 0x40600000)
        self.dec = 10
    def start(self, dec_log2=10):
        self.dec = dec_log2
        self.w(0x00, 0)
        self.w(0x08, dec_log2)
        self.w(0x0C, 0)
        self.w(0x00, 1 | 2)
        self.w(0x00, 1)
    def tune(self, f):
        self.w(0x04, int(round(f / FS_CLK * (1 << 32))) & 0xFFFFFFFF)
    def measure(self, navg=N_AVG):
        zi = 0.0; zq = 0.0
        for _ in range(navg):
            zi += s32(self.r(0x18))
            zq += s32(self.r(0x1C))
        return complex(zi / navg, zq / navg)
    def f_out(self):
        return FS_CLK / (1 << self.dec)


class Resonator(Bus):
    def __init__(self):
        Bus.__init__(self, 0x40400000)
        self.f0_q = None; self.q_q = None
    # Output shift, as an ATTRIBUTE rather than a default argument. It used
    # to be hard-coded 7 here AND again in shift_f0 below, so retuning the
    # coil mid-scan silently put the shift back to 7 and moved the whole
    # gain axis. One place decides it now, and the CONFIG cell sets it.
    #
    # Not a module-level global read inside config(): that is a forward
    # reference to a name the CONFIG cell has not defined yet when this
    # cell runs, and verify_notebooks.py rightly refuses it. An attribute
    # with a default here is defined before anything can call it.
    oshift = 7

    def config(self, f0, Q, delay=15, oshift=None):
        if oshift is None:
            oshift = self.oshift
        w0 = 2.0 * np.pi * f0 / FS_CLK
        r  = 1.0 - np.pi * (f0 / FS_CLK) / Q
        ci = int(round(r * np.cos(w0) * 131072))
        cs = int(round(r * np.sin(w0) * 131072))
        self.w(0x04, ci & 0x3FFFF)
        self.w(0x08, cs & 0x3FFFF)
        self.w(0x0C, delay)
        self.w(0x10, oshift)
        rq        = np.hypot(ci, cs) / 131072.0
        self.f0_q = np.arctan2(cs, ci) / (2.0 * np.pi) * FS_CLK
        self.q_q  = np.pi * (self.f0_q / FS_CLK) / (1.0 - rq)
    def shift_f0(self, df, Q=None):
        """Retune the emulated coil by df hertz - the drift the servo fights.
        One coefficient rewrite; Q is preserved."""
        self.config(self.f0_q + df, Q if Q is not None else self.q_q,
                    delay=15)          # inherits OSHIFT, see config()
    # --- the two noise knobs (registers added for this stage) -------------
    def noise_tank(self, shift=None):
        """Johnson noise INTO the loop -> Lorentzian bump. None = off.
        Each shift step is 6 dB down from raw (~9460 counts rms)."""
        self.w(0x20, 0 if shift is None else (0x100 | (shift & 0x1F)))
    def noise_out(self, shift=None):
        """Amplifier noise AFTER the loop -> flat floor. None = off."""
        self.w(0x24, 0 if shift is None else (0x100 | (shift & 0x1F)))
    def enable(self):  self.w(0x00, 1)
    def disable(self): self.w(0x00, 0)
    def clear(self):   self.w(0x00, 3)
    def clipped(self): return bool(self.r(0x14) & 1)
    def peek(self):    return s32(self.r(0x18)), s32(self.r(0x1C))


class PhaseShift(Bus):
    LATENCY = 2
    COEF_FS = 16384.0                 # coefficient full scale -> gain LSB
    def __init__(self):
        Bus.__init__(self, 0x40300000)
    def config_delay(self, f):
        best = None
        for k in range(0, 9):
            D = FS_CLK / (4.0 * f) + k * FS_CLK / f
            if D < 1.0 or D > 1000:
                continue
            frac  = D - np.floor(D)
            score = min(frac, 1.0 - frac)
            if best is None or score < best[0]:
                best = (score, int(np.floor(D)), frac)
        self.w(0x08, best[1])
        self.w(0x0C, int(round(best[2] * 16384)))
        self.w(0x10, 0)
        return best[1], best[2]
    def set(self, A, phi_deg, f=F0):
        phi = phi_deg - 360.0 * self.LATENCY * f / FS_CLK
        self.w(0x00, int(round(A * np.cos(np.radians(phi)) * self.COEF_FS)) & 0xFFFF)
        self.w(0x04, int(round(A * np.sin(np.radians(phi)) * self.COEF_FS)) & 0xFFFF)
    def gain_lsb(self, phi_deg, f=F0):
        """Smallest change in A this actuator can express at this phase. The
        two coefficients quantise independently, so the step in |A| is set by
        the LARGER of the two projections. Step 5c compares this against the
        width of the band a software loop would need to steer within."""
        phi = np.radians(phi_deg - 360.0 * self.LATENCY * f / FS_CLK)
        return 1.0 / (self.COEF_FS * max(abs(np.cos(phi)), abs(np.sin(phi))))
    def off(self):
        self.set(0.0, 0.0)


class Generator(Bus):
    def __init__(self):
        Bus.__init__(self, 0x40200000, span=0x20000)
    def load_sine(self):
        for i in range(16384):
            v = int(round(8191.0 * np.sin(2.0 * np.pi * i / 16384.0)))
            self.w(0x10000 + 4 * i, v & 0x3FFF)
        self.w(0x08, (16384 << 16) - 1)
        self.w(0x0C, 0)
    def amp(self, frac):
        self.w(0x04, int(round(frac * 8192)) & 0x3FFF)
    def freq(self, f):
        self.w(0x10, int(round(f / FS_CLK * (1 << 30))))
    def run(self):
        self.w(0x00, 0x40)
        self.w(0x00, 0x11)
    def stop(self):
        self.amp(0.0)
        self.w(0x00, 0xC0)


acq, reso, ps, gen = Acq(), Resonator(), PhaseShift(), Generator()
print('drivers ready  (numpy %s, matplotlib %s)'
      % (np.__version__, matplotlib.__version__))

In [ ]:
# =============================================================================
#  Memory capture - the Stage 6/7 system, condensed. Noise SPECTRA need time
#  records, and time records need memory; the PEEK voltmeter cannot see a
#  spectrum. Same safeguards as Stage 7: region from the device tree, stop
#  address inclusive, poison pre-fill, write-error and poison checks that
#  RAISE rather than return plausible garbage. (Bitstreams from the Stage 8
#  rebuild onward carry the trusted badge, so TrustZone no longer matters -
#  but the checks stay, because they cost nothing and catch everything.)
# =============================================================================
def _be32(b, i=0):
    return struct.unpack('>I', b[i * 4:i * 4 + 4])[0]

def _dt_read(path):
    for root in ('/sys/firmware/devicetree/base', '/proc/device-tree'):
        try:
            with open(os.path.join(root, path), 'rb') as fh:
                return fh.read()
        except (IOError, OSError):
            pass
    return None

blob = _dt_read('memory/reg') or _dt_read('memory@0/reg')
assert blob and len(blob) >= 8, 'cannot read the device tree memory node'
declared_end = _be32(blob, 0) + _be32(blob, 1)
assert declared_end < 512 * 1024**2, 'no memory above the declared end - see Stage 6 Step 1'
MEM_START = (declared_end + 0xFFF) & ~0xFFF
MEM_STOP  = MEM_START + (0x00800000 & ~7) - 8       # LAST address, inclusive
POISON    = 0xDEADBEEF
print('capture region 0x%08X .. 0x%08X inclusive' % (MEM_START, MEM_STOP + 7))


CAPTURE_TRACE = None      # set to a path (e.g. '/tmp/capture.log') to log every
                          # capture to disk before it runs


def _trace(msg):
    """Append to the trace file and force it to disk.

    A dead kernel takes buffered stdout with it, which is why a crash inside
    capture() leaves the notebook showing nothing at all. This survives,
    so the last line in the file names the call that killed it."""
    if not CAPTURE_TRACE:
        return
    with open(CAPTURE_TRACE, 'a') as fh:
        fh.write('%.3f %s\n' % (time.time(), msg))
        fh.flush()
        os.fsync(fh.fileno())


def _poison(nbytes):
    fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
    mm = mmap.mmap(fd, nbytes, offset=MEM_START)
    try:
        mm[0:nbytes] = np.full(nbytes // 4, POISON, dtype='<u4').tobytes()
    finally:
        mm.close(); os.close(fd)


def capture(f_lo, dec_log2, n_samples, settle=SETTLE):
    """Record n_samples of the complex stream; raises on any transport fault."""
    _trace('capture f=%.1f dec=%d n=%d' % (f_lo, dec_log2, n_samples))
    _poison(n_samples * 8)
    acq.w(0x00, 0)
    acq.w(0x08, dec_log2); acq.dec = dec_log2
    acq.w(0x0C, 0)
    acq.tune(f_lo)
    acq.w(0x28, MEM_START)
    acq.w(0x2C, MEM_STOP)
    # Read the bounds back BEFORE arming. The writer is a DMA master: if the
    # stop bound did not take - wrong bitstream, unimplemented register, a
    # write that silently went nowhere - it runs from MEM_START upward through
    # whatever follows, and what follows is the memory Linux is using. That is
    # not an exception with a traceback to read; it is a dead kernel. Two
    # register reads cost microseconds and turn the worst failure mode in this
    # notebook into a sentence.
    lo_rb, hi_rb = acq.r(0x28), acq.r(0x2C)
    if lo_rb != MEM_START or hi_rb != MEM_STOP:
        acq.w(0x00, 0)                              # ensure nothing is armed
        raise RuntimeError(
            'capture bounds did not take: wrote 0x%08X..0x%08X, read back '
            '0x%08X..0x%08X. Refusing to arm an unbounded DMA writer.'
            % (MEM_START, MEM_STOP, lo_rb, hi_rb))
    if n_samples * 8 > (MEM_STOP + 8 - MEM_START):
        raise ValueError('record of %d samples (%d bytes) exceeds the %d-byte '
                         'capture region'
                         % (n_samples, n_samples * 8, MEM_STOP + 8 - MEM_START))
    # SETTLE FIRST, RECORD SECOND. Arming immediately and sleeping afterwards
    # would let the record's opening milliseconds hold the filter pipeline still
    # flushing the PREVIOUS tuning - which in a sweep poisons each point with its
    # predecessor's power, most visibly the first point, and can drag a fit wide
    # by tens of percent from the far end of the grid. Instead the chain
    # free-runs at the new tuning while the pipeline flushes, and only then does
    # the writer start: sample 0 of every record is already settled.
    acq.w(0x00, 1 | 2)                              # run at the new tuning...
    acq.w(0x00, 1)
    time.sleep(settle)                              # ...flush the pipeline...
    acq.w(0x00, 1 | 16 | 128)                       # ...then arm, fresh writer
    acq.w(0x00, 1 | 16)
    # WAIT FOR THE WRITER; DO NOT GUESS AT IT.
    #
    # This used to be one sleep of "as long as the record should take, plus
    # 5 ms", after which the writer was disarmed and memory read. Two things
    # were wrong with that, and both showed up on 2026-08-04 as an intermittent
    # 'unwritten words in the record' on captures that were perfectly healthy:
    #
    #  1. DMA_COUNT (0x34) was not consulted until AFTER the decision to stop
    #     had already been made. The sleep was a guess with no verification.
    #  2. DMA_COUNT reports samples HANDED TO the memory writer, not samples
    #     that have landed in DDR. The writer bursts, so the last burst can
    #     still be in flight when the count already reads complete.
    #
    # A fixed 5 ms of margin is also the wrong shape: it is 30% of a short
    # record and 0.1% of a long one, so the failure is rare, load-dependent and
    # looks like a hardware fault. It is not - the record was read a few
    # hundred microseconds early.
    #
    # Poll until the record is complete, then let the last burst land.
    t_record = n_samples * (1 << dec_log2) / FS_CLK
    deadline = time.time() + t_record + 0.5         # generous, not 5 ms
    while int(acq.r(0x34)) < n_samples and time.time() < deadline:
        time.sleep(0.001)
    acq.w(0x00, 1)                                  # disarm
    time.sleep(0.002)                               # last burst reaches DDR

    st  = acq.r(0x38)
    if st & 8:
        raise RuntimeError('memory refused writes - see Stage 6; run check_system.py')
    got = min(int(acq.r(0x34)), n_samples)
    # try/finally, because everything between the open and the close can raise:
    # mmap itself, the slice (SIGBUS on a bad physical address), an allocation
    # failure on a tight board. Without it each failure leaks a file descriptor
    # AND a mapping of /dev/mem - and Stage 10's scan loop swallows per-point
    # exceptions by design, so the leak would repeat silently across hundreds
    # of grid points until the process runs out of descriptors or address
    # space. That surfaces later as an unrelated-looking out-of-memory error.
    def _read_region(nbytes):
        fd = mm = None
        try:
            fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
            mm = mmap.mmap(fd, ((nbytes + 8191) // 4096) * 4096, offset=MEM_START)
            return mm[0:nbytes]
        finally:
            if mm is not None:
                mm.close()
            if fd is not None:
                os.close(fd)

    buf = _read_region(got * 8)
    u32 = np.frombuffer(buf, dtype='<u4')
    _limit = max(8, 0.001 * u32.size)
    if got and int(np.count_nonzero(u32 == POISON)) > _limit:
        # A word still inside the writer's burst buffer is not a lost word. If
        # it is genuinely lost it will still be poison in 20 ms; if it was
        # merely late it will not. Retrying once separates the two, which the
        # single-shot check could not: it reported a transport fault for both.
        time.sleep(0.02)
        buf = _read_region(got * 8)
        u32 = np.frombuffer(buf, dtype='<u4')
        n_bad = int(np.count_nonzero(u32 == POISON))
        if n_bad > _limit:
            raise RuntimeError(
                'unwritten words in the record - transport fault, see Stage 6 '
                '(%d of %d words still unwritten 20 ms after the writer '
                'reported the record complete)' % (n_bad, u32.size))
    raw = np.frombuffer(buf, dtype='<i4').astype(np.float64)
    acq.w(0x00, 1)                                  # back to free-running voltmeter
    _trace('   ok got=%d' % got)
    return raw[0::2] + 1j * raw[1::2]


def welch_psd(z, seg=2048):
    """Averaged power spectrum of a complex record - the standard way to see
    noise: many short transforms averaged beat one long one, because noise in
    a single transform is 100% uncertain no matter how long it is."""
    k = len(z) // seg
    w = np.hanning(seg)
    p = np.zeros(seg)
    for i in range(k):
        p += np.abs(np.fft.fft(z[i * seg:(i + 1) * seg] * w)) ** 2
    f = np.fft.fftfreq(seg, 1.0 / acq.f_out())
    order = np.argsort(f)
    return f[order], p[order] / max(k, 1)


def _lorentz_core(f, p, guard=0.05, minpts=3):
    """Centre, width, peak and BASELINE of a Lorentzian power curve.

    1/p is a parabola in f only for a pure Lorentzian. Any additive floor -
    detector noise, an amplifier pedestal - is absorbed into that parabola and
    inflates the fitted width. That is negligible where the peak towers over
    the floor and severe where it does not, because peak power falls as the
    square of the Q ratio: a floor worth 2% of the open-loop peak is worth 32%
    of a 4x-broadened one. Uncorrected bias, measured: +18% in linewidth at
    Q_eff/Q0 = 1, +26% at 0.5, +153% at 0.25.

    The floor is found by search, not from the wings. For each trial floor:
    subtract it, fit, rebuild the full Lorentzian-plus-floor, and keep the
    trial that reproduces the measured points best. A wing estimate needs
    points beyond ~1.5 linewidths, which a broad line does not have on this
    grid - exactly the case that needs correcting most. The search uses every
    point, so it works at any width. Two passes: coarse over [0, min(p)], then
    fine around the winner.

    Returns (f0, linewidth, peak_above_baseline, baseline).
    """
    f = np.asarray(f, dtype=float)
    p = np.asarray(p, dtype=float)
    pmax = float(np.max(p))
    u = f - f.mean()

    def _try(b):
        q = p - b
        keep = q > guard * q.max()
        if int(np.count_nonzero(keep)) < minpts:
            return None
        A, B, C = np.polyfit(u[keep], 1.0 / q[keep], 2, w=q[keep])
        if A <= 0.0:
            return None                    # parabola opens the wrong way
        u0 = -B / (2.0 * A)
        lw = 2.0 * np.sqrt(max(C / A - u0 ** 2, 1e-30))
        p0 = 1.0 / max(C - A * u0 ** 2, 1e-300)
        f0 = f.mean() + u0
        mod = p0 / (1.0 + (2.0 * (f - f0) / lw) ** 2) + b
        return float(np.sqrt(np.mean(((p - mod) / pmax) ** 2))), (f0, lw, p0, b)

    hi = 0.999 * float(np.min(p))
    best, best_b = _try(0.0), 0.0           # b = 0 is the uncorrected fit
    if hi > 0.0:
        for b in np.linspace(0.0, hi, 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
        w = hi / 16.0
        for b in np.linspace(max(best_b - w, 0.0), min(best_b + w, hi), 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
    if best is None:
        raise ValueError('fit window too narrow: %d points'
                         % int(np.count_nonzero(p > guard * p.max())))
    f0, lw, p0, base = best[1]
    return f0, lw, p0, base


def fit_lorentzian_psd(f, p, guard=0.05):
    """Centre, width from a power spectrum: 1/p is a parabola (Stage 8), after
    the noise floor has been taken out of p."""
    f0, lw, _, _ = _lorentz_core(f, p, guard=guard, minpts=3)
    return f0, lw

In [ ]:
# ============================ CONFIGURATION ==================================
EMULATION  = True    # True: digital resonator, known answers, PASS gates.
                     # False: the real coil - same scan, gates become measurements

TANK_SH    = 3       # tank (Johnson) noise level, 6 dB per step
OUT_SH     = 4       # amplifier-floor level (off during sweeps automatically)

# The gain axis must BRACKET the self-oscillation boundary, or there is no
# boundary in the atlas and every stability margin the designer prints is
# meaningless. K rises ~0.73 per unit A, so onset is near A = 1.4 (+3 dB).
# Feedback gain axis, in dB of A.
#
# THE TOP OF THIS AXIS IS NOT A CHOICE. The phase shifter's coefficients are
# signed Q2.14, so the largest A that can be represented is 1.99994, which is
# +6.02 dB. The axis already ends there. "Raise the top and rerun" - which two
# messages in this notebook used to advise - cannot work: the actuator has no
# more to give. Reaching further needs a wider coefficient format in the RTL,
# or a plant that goes unstable at lower A (higher Q, or less output shift).
#
# So the black self-oscillating region is bounded by physics at about 3 dB wide:
# onset near A = 1.4 (+3 dB) up to the ceiling at +6 dB. At the old uniform 2 dB
# step that is TWO rows, which is why it appeared as a thin sliver rather than a
# region. The band is not being enlarged here - it is being RESOLVED, by
# spending the sample budget where the interesting behaviour is.
#
#   below 0 dB : 2.0 dB steps - far from onset, nothing changes quickly
#   above 0 dB : 0.5 dB steps - through onset and up to the ceiling
#
# 7 rows inside the unstable band instead of 2. Note the axis is deliberately
# NON-UNIFORM, so the heatmaps use pcolormesh rather than imshow - imshow
# assumes evenly spaced rows and would silently draw them in the wrong places.
# =============================================================================
#  LOOP-GAIN CALIBRATION  -  where self-oscillation lands on the gain axis
#
#  The digital resonator divides its state by 2^OSHIFT before the output leaves
#  the module, so the loop gain scales exactly as 2^-OSHIFT. Nothing else in the
#  loop is adjustable, which makes this the one constant that decides where
#  onset falls on the axis below.
#
#  WHY IT CHANGED FROM 7 TO 5
#  With OSHIFT = 7 the fitted loop gain was k = 0.73 per unit A, putting onset
#  at A = 1/0.73 = 1.37, i.e. +2.73 dB. The actuator ceiling is +6.02 dB
#  (coefficients are signed Q2.14, so A <= 1.99994), which left only 3.3 dB of
#  usable range above onset. That is why the self-oscillating region came out as
#  a sliver: it was not small, it was CUT OFF by the ceiling.
#
#  Herkenhoff et al. Fig. 9 has onset at about -15.5 dB with 5.5 dB of axis
#  above it. Two steps of OSHIFT move onset by 2 x 6.02 = 12.04 dB:
#
#      OSHIFT   k = 0.73 x 2^(7-OSHIFT)   predicted onset
#         7            0.73                   +2.73 dB   (was: 3.3 dB of room)
#         6            1.46                   -3.29 dB
#         5            2.92                   -9.31 dB   <- now
#         4            5.84                  -15.33 dB   (the paper's onset)
#
#  OSHIFT = 5 is chosen over 4 deliberately. It gives a paper-like picture while
#  leaving 15 dB of actuator margin above onset instead of 21, and the margin
#  matters: lowering OSHIFT makes the resonator's output larger, and the OUTPUT
#  saturator in red_pitaya_resonator.v HAS NO STATUS FLAG. Only the state
#  overflow sets `clipped`. Clip the output and nothing tells you - the loop
#  gain simply compresses and onset drifts upward, which reads as a wrong
#  measurement rather than a fault.
#
#  Two existing checks catch that, and both must be believed:
#    * the open-loop assert in this cell (Q_OL within 10% of reso.q_q) - a
#      clipped output distorts the lineshape and moves the fitted Q
#    * Q_LIN in Step 1 - the loop-linearity check across two drive levels
#
#  NOTE FOR OTHER STAGES: this changes the plant's output level by 12 dB
#  relative to Stages 8, 9 and 11, which still use OSHIFT = 7. Any feedback
#  setting carried between notebooks has to be rescaled by the same 12 dB, or
#  re-derived. Do not compare absolute A values across stages without it.
OSHIFT = 5
Resonator.oshift = OSHIFT        # every config() and shift_f0() inherits it

# Feedback gain axis, in dB of A.
#
# Framed like the paper's: roughly 10 dB below the predicted onset to 5 dB above
# it, rather than running to the actuator ceiling. Deliberately wider than that
# on the low side, because the onset prediction above is an extrapolation from a
# k fitted at OSHIFT = 7 - if it is off by a few dB the axis must still bracket
# the real thing, or the scan is wasted.
#
# Coarse far from onset, fine through it: the sample budget belongs where the
# behaviour changes. The axis is NON-UNIFORM, so the heatmaps use pcolormesh -
# imshow assumes evenly spaced rows and would draw them in the wrong places.
GAINS_DB   = np.concatenate([np.arange(-24.0, -14.0, 2.0),
                             np.arange(-14.0,  -2.01, 0.5)])
PHIS       = np.arange(0.0, 360.0, 15.0)     # feedback phase axis: 24 points
SWEEP_PTS  = 15      # frequencies per mini-sweep at each grid point
SWEEP_SPAN = 2.5     # mini-sweep half-width in open-loop linewidths
SWEEP_N    = 2048    # samples per frequency point (halved at deep decimations)
SETTLE     = 0.02

# A point is OSCILLATING if, with the noise drive removed, it still holds this
# fraction of its driven power. A ratio, so it needs no absolute floor - on a
# digital plant the undriven floor is exactly zero.
SUS_OSC    = 0.15
DRIVE_STEP = 2       # noise-shift steps (12 dB) used to test loop linearity
# Grid cost: 26 x 24 points x ~1 s. About 10 minutes.
# =============================================================================

# --- Step 0: right bitstream, plant configured, knob calibrated --------------
reso.w(0x20, 0x105)
assert (reso.r(0x20) & 0x1FF) == 0x105, 'noise registers absent - rebuild (see Stage 9 Step 0)'
reso.w(0x20, 0)

reso.config(F0, Q_SET)                       # oshift comes from OSHIFT
ps.config_delay(F0); ps.off()
gen.load_sine(); gen.amp(0.0); gen.run()
reso.enable(); reso.clear()
acq.start(dec_log2=10); acq.tune(reso.f0_q)
time.sleep(0.2)

# --- does the gain axis bracket where onset is expected to be? ---------------
#  A 12-minute scan that misses the boundary produces an atlas with no boundary
#  in it, and every stability margin printed from it is then meaningless. The
#  prediction is an extrapolation - k was fitted at OSHIFT = 7 - so this warns
#  rather than asserts, but it warns before the 12 minutes are spent.
K_PER_A_AT_OSHIFT7 = 0.73                    # Step 3 of the previous scan
_k_pred   = K_PER_A_AT_OSHIFT7 * 2.0 ** (7 - OSHIFT)
_on_pred  = 20.0 * np.log10(1.0 / _k_pred)
print('OSHIFT %d  ->  k = %.2f per unit A  ->  onset predicted at %+.2f dB'
      % (OSHIFT, _k_pred, _on_pred))
print('gain axis %+.1f to %+.1f dB, actuator ceiling %+.2f dB'
      % (GAINS_DB[0], GAINS_DB[-1], 20 * np.log10(2 - 2 ** -14)))
if not (GAINS_DB[0] < _on_pred < GAINS_DB[-1]):
    print('  WARNING: the predicted onset is OUTSIDE the gain axis. The scan will')
    print('  find no boundary and every margin computed from it will be empty.')
    print('  Move OSHIFT or GAINS_DB before spending 12 minutes on this.')
else:
    print('  onset sits %.1f dB above the bottom of the axis and %.1f dB below the'
          % (_on_pred - GAINS_DB[0], GAINS_DB[-1] - _on_pred))
    print('  top, with %.1f dB of actuator margin left above the axis.'
          % (20 * np.log10(2 - 2 ** -14) - GAINS_DB[-1]))
print()
z_bg = acq.measure(64)
LW0  = reso.f0_q / reso.q_q


def coherence(z):
    """1 for pure noise, 0 for a pure tone; equals 1 - f^2 where f is the
    power fraction carried by a coherent tone. Reported as a diagnostic.
    Not used to classify points: for narrowband noise the effective sample
    count is small and this scatters ~30%, far too coarse to detect anything
    subtle."""
    p = np.abs(z) ** 2
    m = float(np.mean(p))
    return float(np.var(p)) / max(m * m, 1e-300)


def tone_fraction(R):
    return float(np.sqrt(max(0.0, 1.0 - R)))


def fit_lorentz3(f, p, guard=0.10):
    """Centre, width, peak and baseline of a Lorentzian power curve (the
    Stage 9 method, with the floor removed - see _lorentz_core in the cell
    above). Raises if the guard leaves too few points to determine a parabola;
    silence there is how a thin fit window becomes a huge Q.

    The peak is now the height ABOVE the baseline, so (pi/2)*p0*lw is the area
    of the bump alone - which is what T_eff wants. The baseline is returned so
    the caller can rebuild the full curve when testing the fit."""
    return _lorentz_core(f, p, guard=guard, minpts=5)


def _win_half(dec):
    """The receiver's listening half-width at decimation 2^dec."""
    return 0.27 * FS_CLK / (1 << dec)


def _dec_for(spacing):
    """Match the receiver to the grid: window ~ half the point spacing,
    rounded WIDER. A too-wide window smears the width, which the
    deconvolution below undoes exactly; a too-narrow one leaves gaps a narrow
    line can hide in, which no amount of fitting can undo."""
    d = int(np.floor(np.log2(0.27 * FS_CLK / (0.5 * spacing))))
    return int(np.clip(d, 8, 14))


def _model(fs, dec, f0, lw):
    """A width-lw Lorentzian at f0 seen through this receiver on this grid:
    the window-averaged (smeared) curve, in closed form."""
    W = _win_half(dec)
    return ((lw / 2.0) * (np.arctan(2.0 * (fs + W - f0) / lw)
                        - np.arctan(2.0 * (fs - W - f0) / lw)) / (2.0 * W))


def _deconv_lw(lw_m, f0, fs, dec):
    """Undo the window smear with no calibration constants: bisect the true
    width whose model, pushed through the SAME window, grid and fit,
    reproduces the measured width.

    Returns (lw_true, bracketed). `bracketed` means the search hit its lower
    bound - the correction wanted to be more than 4x and was truncated, which
    would inflate Q with no other symptom. The caller discards those."""
    def fwd(lw_t):
        # The guard in fit_lorentz3 protects fits to real DATA. Tripping it on
        # our own noise-free model just means the trial width is too narrow for
        # this grid to express, which is the same as "narrower than measured".
        try:
            return fit_lorentz3(fs, _model(fs, dec, f0, lw_t) + 1e-12)[1]
        except ValueError:
            return 0.0
    lo, hi = lw_m / 4.0, lw_m * 1.05
    if fwd(hi) < lw_m:
        return lw_m, False               # smear negligible at this geometry
    for _ in range(30):
        mid = 0.5 * (lo + hi)
        if fwd(mid) < lw_m:
            lo = mid
        else:
            hi = mid
    out = 0.5 * (lo + hi)
    return out, bool(out < 1.02 * lw_m / 4.0)


def noise_sweep(centre, pts=SWEEP_PTS):
    """Adaptive power sweep. Each pass matches the receiver to the grid, then
    reads the data before trusting any fit:
      - low contrast          -> structure wider than the span: widen, retry
      - one point owns the power -> narrower than the grid: zoom on it
      - otherwise fit, accepting only if the centre is inside the span, the
        width is resolvable here, and the curve goes through the points
    An accepted width is deconvolved from the receiver window. Five passes
    maximum; anything still unresolved returns nans - reported, never stored.

    Returns (f0, lw_true, p0, area, coh). Area uses the MEASURED (smeared)
    peak and width: window averaging conserves the integral, so that pair
    gives the true area directly, while Q needs the deconvolved width."""
    span = SWEEP_SPAN * LW0
    for _ in range(5):
        fs = centre + np.linspace(-span, span, pts)
        spacing = fs[1] - fs[0]
        dec = _dec_for(spacing)
        nsamp = SWEEP_N if dec <= 12 else SWEEP_N // 2
        p  = np.empty(pts); ch = np.empty(pts)
        for k, fq in enumerate(fs):
            z = capture(fq, dec, nsamp)
            p[k]  = float(np.var(z))
            ch[k] = coherence(z)
        coh   = float(ch[int(np.argmax(p))])
        base  = float(np.min(p))
        share = (p.max() - base) / max(float(np.sum(p - base)), 1e-300)
        contrast = p.max() / max(float(np.median(p)), 1e-300)
        if contrast < 2.0:
            span = min(span * 3.0, 8.0 * LW0)
            continue
        if share > 0.6:
            centre = float(fs[int(np.argmax(p))])
            span   = max(span / 5.0, 5e2)
            continue
        try:
            f0, lw, p0, pbase = fit_lorentz3(fs, p)
        except ValueError:
            centre = float(fs[int(np.argmax(p))])
            span   = max(span / 5.0, 5e2)
            continue
        inside = bool(fs[0] < f0 < fs[-1])
        # The baseline must be added back before judging the fit: comparing
        # data-with-floor against a floor-free Lorentzian rejects a CORRECT fit
        # whenever the floor matters, which is exactly the broad-line case the
        # baseline removal was added for.
        fitc   = p0 / (1.0 + (2.0 * (fs - f0) / lw) ** 2) + pbase
        resid  = float(np.sqrt(np.mean(((p - fitc) / p.max()) ** 2)))
        if inside and 1.5 * spacing <= lw <= (fs[-1] - fs[0]) and resid < 0.15:
            area = (np.pi / 2.0) * p0 * lw
            lw_t, bracketed = _deconv_lw(lw, f0, fs, dec)
            if bracketed:
                return np.nan, np.nan, np.nan, np.nan, coh
            return f0, lw_t, p0, area, coh
        if inside and spacing / 5.0 < lw < 1.5 * spacing:
            centre, span = f0, max(2.0 * lw, 5e2)
        else:
            centre = float(fs[int(np.argmax(p))])
            span   = max(span / 5.0, 5e2)
    return np.nan, np.nan, np.nan, np.nan, np.nan        # honest failure


reso.noise_tank(TANK_SH); reso.noise_out(None)
reso.clear(); time.sleep(0.05)
F_OL, LW_OL, PK_OL, AREA_OL, COH_OL = noise_sweep(reso.f0_q)
assert np.isfinite(LW_OL), 'open-loop sweep unresolved - something is badly wrong'
Q_OL = F_OL / LW_OL
print('open loop: f0 %.1f Hz, linewidth %.0f Hz, Q %.1f  (known: %.1f)'
      % (F_OL, LW_OL, Q_OL, reso.q_q))
print('           coherence %.2f (want ~1 - pure noise, no tone)' % COH_OL)
if EMULATION:
    assert abs(Q_OL / reso.q_q - 1.0) < 0.10, 'open-loop Q wrong - fix before scanning'
print('STEP 0: ready')

---
## Step 1 — The scan

At every gain/phase pair, two questions.

**Is the loop feeding itself?** Remove the noise drive and look at what is left. A
resonator, however narrow, decays to nothing; an oscillator holds itself up. Comparing
before and after as a *ratio* is what makes this work — the undriven floor of a digital
plant is exactly zero, so any absolute threshold would be tripped by a single bit. The
clip flag alone is not enough: this loop limits gently and parks just below the boundary
without ever saturating.

**If not, what is it doing?** The Stage 9 power sweep and a Lorentzian fit give Q, centre,
peak and area.

Afterwards the scan measures one more thing: **how far it can be trusted.** Drop the drive
12 dB and a linear system's bump area must fall by exactly 16×. Where it stops doing that,
the signal inside the resonator has grown large enough to compress, and the fits above that
point are not measurements. That limit is a single number in Q, since the internal signal
grows as `sqrt(Q)` at fixed drive.

Everything is measured **in noise** — the maps describe the system as the real measurement
will meet it.

> **Real coil — two mechanisms here do not exist on a real coil.**
>
> `probe_point` decides whether a setting is self-oscillating by switching the coil's thermal
> noise **off** and seeing whether the signal survives. Johnson noise cannot be switched off.
> Replace it with the coherence measure `R = var(|z|²)/mean(|z|²)²`, already defined above: it
> is 1 for pure noise and 0 for a pure tone, so an oscillation reads near 0 and a thermal bump
> near 1. That distinction is large and easy — it is only *small* differences that R measures
> poorly.
>
> The linear-limit calibration drops the tank noise by 12 dB and checks the bump area falls
> 16×. You cannot change a real coil's thermal drive either. Inject a known tone through the
> generator at two amplitudes and check the measured response scales in proportion instead.


In [ ]:
GAINS = 10.0 ** (GAINS_DB / 20.0)
NG, NP = len(GAINS), len(PHIS)
M_Q   = np.full((NG, NP), np.nan)      # quality factor
M_DF  = np.full((NG, NP), np.nan)      # centre shift [Hz]
M_T   = np.full((NG, NP), np.nan)      # area ratio = T_eff / T0
M_CH  = np.full((NG, NP), np.nan)      # coherence (diagnostic)
M_SUS = np.full((NG, NP), np.nan)      # self-sustained fraction


def probe_point(A, phi):
    """How much of this setting's output is the loop feeding itself?

    Returns p_off/p_on - the power left when the noise drive is removed, as a
    fraction of the driven power. A passive resonator decays to nothing, so
    the ratio is ~1e-5 however narrow it is; an oscillator holds itself up, so
    the ratio is ~1. This is the only test that works here: the clip flag
    catches a HARD runaway, but this loop limits softly and parks just below
    the boundary without ever saturating anything.

    Deliberately a ratio, not a comparison against a floor - the undriven
    floor of a digital plant is exactly zero, so any absolute threshold is
    satisfied by a single least-significant bit.

    capture() settles 20 ms before recording, which is ~14000 ring-down time
    constants, so even a marginal point is fully decayed by sample 0."""
    ps.set(A, phi)
    reso.clear()
    time.sleep(0.010)
    if reso.clipped():
        return 1.0                                # hard runaway
    p_on = float(np.var(capture(F_OL, 10, 512)))
    reso.noise_tank(None)
    p_off = float(np.var(capture(F_OL, 10, 512)))
    reso.noise_tank(TANK_SH)
    reso.clear()
    return p_off / max(p_on, 1e-300)


t0 = time.time()
n_err = 0
for i, A in enumerate(GAINS):
    for j, phi in enumerate(PHIS):
        sus = probe_point(A, phi)
        M_SUS[i, j] = sus
        if sus < SUS_OSC:
            try:
                fc, lw, pk, area, coh = noise_sweep(reso.f0_q)
                M_CH[i, j] = coh
                if np.isfinite(lw):               # nans never become data
                    M_Q[i, j]  = fc / lw
                    M_DF[i, j] = fc - F_OL
                    M_T[i, j]  = area / AREA_OL
            except RuntimeError:
                pass                              # transport fault: leave nan
            except Exception as exc:
                # One bad point must never cost the other 407.
                n_err += 1
                if n_err <= 5:
                    print('    point (%+.1f dB, %3.0f deg) failed: %s: %s'
                          % (GAINS_DB[i], phi, type(exc).__name__, exc))
        ps.off(); reso.clear()
    print('  gain %+6.1f dB done  (%d/%d rows, %.0f s elapsed)'
          % (GAINS_DB[i], i + 1, NG, time.time() - t0))
ps.off()

STAB = M_SUS < SUS_OSC

# --- where does the loop stop being linear? ----------------------------------
# Compression depends on how big the signal inside the resonator gets, and that
# grows as sqrt(Q_eff) at fixed drive - so the limit is a single number in
# Q_eff, the same at every phase. Measure it instead of guessing: drop the
# drive by 12 dB and the bump area must fall by exactly 16x. Walk down from the
# strongest gain and take the first point that passes.
j_hi  = int(np.nanargmax(np.nanmax(np.where(STAB, M_Q, np.nan), axis=0)))
Q_LIN = 0.0                      # nothing trusted until something passes
want  = 4.0 ** DRIVE_STEP
for i in range(NG - 1, -1, -1):
    if not (STAB[i, j_hi] and np.isfinite(M_Q[i, j_hi])):
        continue
    ps.set(GAINS[i], PHIS[j_hi]); reso.clear()
    reso.noise_tank(TANK_SH);              time.sleep(0.02)
    a_hi = noise_sweep(reso.f0_q)[3]
    reso.noise_tank(TANK_SH + DRIVE_STEP); time.sleep(0.02)
    a_lo = noise_sweep(reso.f0_q)[3]
    reso.noise_tank(TANK_SH); ps.off(); reso.clear()
    if np.isfinite(a_hi) and np.isfinite(a_lo):
        if abs((a_hi / a_lo) / want - 1.0) < 0.15:
            Q_LIN = M_Q[i, j_hi]
            break

# Default to trusting NOTHING, not everything. If no point on the column
# passes, the drive is too high for this plant even at the bottom of the gain
# axis, and the whole scan is a nonlinear measurement - which must fail loudly,
# not quietly widen the trusted region to include every point.
LINEAR = STAB & np.isfinite(M_Q) & (M_Q <= Q_LIN)

n_unst  = int((~STAB).sum())
n_unres = int((STAB & ~np.isfinite(M_Q)).sum())
n_comp  = int((STAB & np.isfinite(M_Q) & ~LINEAR).sum())
print()
if Q_LIN > 0:
    print('loop stays linear up to Q_eff = %.0f (x%.1f open loop); above that the'
          % (Q_LIN, Q_LIN / Q_OL))
    print('signal inside the resonator is large enough to compress, and the fits')
    print('there are not measurements. %d point(s) affected.' % n_comp)
else:
    print('THE LOOP IS NONLINEAR EVERYWHERE. Dropping the drive %d dB did not'
          % (6 * DRIVE_STEP))
    print('change the bump area by the %.0fx a linear system must give, at ANY'
          % want)
    print('gain on the strongest column - so nothing in this scan is a')
    print('measurement. Raise TANK_SH (quieter drive) and re-run from Step 0.')
print()
print('scan complete: %d points, %d oscillating (%.0f%%), %d unresolved, '
      '%d compressed, %d failed, %.0f s'
      % (NG * NP, n_unst, 100.0 * n_unst / (NG * NP), n_unres, n_comp, n_err,
         time.time() - t0))
ok1 = (STAB.sum() > 0.5 * NG * NP and n_unst > 0
       and n_err < 0.02 * NG * NP and Q_LIN > 0)
if n_unst == 0:
    print('  No oscillating point found. Two different things this can mean,')
    print('  and the axis top tells you which:')
    print('    axis ends at %+.1f dB, actuator ceiling is %+.2f dB'
          % (GAINS_DB[-1], 20 * np.log10(2 - 2 ** -14)))
    if GAINS_DB[-1] < 20 * np.log10(2 - 2 ** -14) - 0.5:
        print('  There IS actuator range left above the axis. Either raise the top')
        print('  of GAINS_DB, or lower OSHIFT by one step to move onset down by')
        print('  6.02 dB - the second is usually the better trade, because it')
        print('  keeps the interesting part of the map away from the ceiling.')
    else:
        print('  The axis already reaches the ceiling: coefficients are signed')
        print('  Q2.14, so A = 1.99994 is the most the actuator can represent.')
        print('  There is no more gain to ask for. Lower OSHIFT, raise the plant')
        print('  Q, or widen the coefficient format in red_pitaya_phaseshift.v.')
if n_unst in (0, NG * NP):
    print('  SANITY: every point returned the same verdict. A %.0f dB gain axis'
          % (GAINS_DB[-1] - GAINS_DB[0]))
    print('  cannot be uniformly anything - this is a broken detector, not a')
    print('  measurement. Check probe_point at the bottom-left corner (must be')
    print('  stable) and the top of the narrowing column (must not be).')
print('STEP 1:', 'PASS' if ok1 else
      'FAIL - the map must be mostly stable AND must contain the boundary')

---
## Step 2 — The maps

The paper's Fig. 8, plus the temperature map. Colour is the measured quantity; black is
self-oscillation; white rings mark points too loud to be a linear measurement. The lower
panels are 1D cuts, as in the paper.

Worth looking for: Q rises towards the black boundary at the narrowing phase and collapses
opposite it; the centre-shift map is the same pattern turned a quarter turn; and the
temperature map should look like a copy of the Q map. That resemblance *is* Eq. 8, and
Step 3 tests it properly.

> **Real coil.** Unchanged. These are the paper's Fig. 8 maps plus the temperature map, drawn
> from whatever was measured.


In [ ]:
# -*- coding: utf-8 -*-
# ============================================================================
#  THE ATLAS, IN THE PAPER'S FORMAT  -  plotting only, no re-measurement.
#  Safe to re-run on data already in memory: it reads M_Q, M_DF, M_T, LINEAR,
#  GAINS_DB and PHIS and writes nothing.
# ============================================================================
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FuncFormatter, LogLocator, MaxNLocator

plt.close('all')

# The three gain cuts, placed RELATIVE TO THE MEASURED ONSET rather than at
# fixed dB values. The paper puts one cut inside the oscillating tongue and two
# below it, about 5 dB apart; doing the same here means the cuts stay where they
# are useful when OSHIFT changes, instead of silently sliding off the
# interesting part of the axis or off the grid entirely.
_osc_rows = [i for i in range(len(GAINS_DB)) if not np.isfinite(M_Q[i]).all()]
if _osc_rows:
    _g_on = GAINS_DB[_osc_rows[0]]              # lowest gain that oscillates
    CUT_GAINS_DB = [_g_on + 0.5, _g_on - 2.5, _g_on - 5.5]
else:
    # No boundary on the grid: fall back to the top third, and say so, because
    # cuts through a map with no onset in it mean something different.
    CUT_GAINS_DB = list(GAINS_DB[[-1, -1 - len(GAINS_DB) // 6,
                                  -1 - len(GAINS_DB) // 3]])
    print('note: no oscillating row on the grid - cuts placed by position, not')
    print('      relative to an onset, because there is no onset to refer to.')
CUT_COLOURS     = ['#EECC66', '#EE8866', '#77AADD']
MARK_COMPRESSED = True

# The narrowing phase, read off the highest fully stable gain row.
_full = [i for i in range(len(GAINS_DB)) if np.isfinite(M_Q[i]).all()]
PHASE_REF = PHIS[int(np.nanargmax(M_Q[_full[-1]]))] if _full else 0.0


def _edges(c):
    c = np.asarray(c, dtype=float)
    e = np.empty(c.size + 1)
    e[1:-1] = 0.5 * (c[1:] + c[:-1])
    e[0]    = c[0]  - 0.5 * (c[1] - c[0])
    e[-1]   = c[-1] + 0.5 * (c[-1] - c[-2])
    return e


def _phase_axis(M):
    """Wrap to -180..180 about PHASE_REF, sort, and close the periodic seam."""
    x = (PHIS - PHASE_REF + 180.0) % 360.0 - 180.0
    o = np.argsort(x)
    x, M = x[o], M[:, o]
    return (np.concatenate([x, [x[0] + 360.0]]),
            np.concatenate([M, M[:, :1]], axis=1))


def _freq_unit(vmax):
    """Pick Hz / kHz / MHz so the tick labels stay short.

    This is what was wrong with the first version: the shift on this plant runs
    to +-160 kHz, so in Hz the colourbar read '-160000 -120000 -80000 ...' and
    the labels collided with each other, with the panel letter and with the
    title above them. Nine ticks of six digits will not fit over a 4-inch axis
    in any font. Scale the data instead of shrinking the text."""
    if vmax >= 1e6:  return 1e6, 'MHz'
    if vmax >= 1e3:  return 1e3, 'kHz'
    return 1.0, 'Hz'


def paper_atlas():
    plt.rcParams.update({'font.family': 'serif', 'mathtext.fontset': 'cm',
                         'axes.linewidth': 0.8, 'font.size': 10.5,
                         'xtick.direction': 'out', 'ytick.direction': 'out'})
    x,  Q   = _phase_axis(M_Q)
    _,  DF  = _phase_axis(M_DF)
    _,  LIN = _phase_axis(np.where(LINEAR, 1.0, np.nan))
    scale, unit = _freq_unit(np.nanmax(np.abs(M_DF)))
    DF = DF / scale
    xe, ye = _edges(x), _edges(GAINS_DB)
    deg = FuncFormatter(lambda v, p: u'%d°' % int(round(v)))

    fig = plt.figure(figsize=(9.6, 6.9))
    # More room all round than before: wider gutter between the columns so the
    # right-hand y label cannot reach the left panel, and a taller colourbar
    # strip so its ticks and its title are not in the same place.
    gs = GridSpec(3, 2, height_ratios=[0.052, 1.0, 0.46],
                  hspace=0.16, wspace=0.42,
                  left=0.105, right=0.975, top=0.885, bottom=0.085)

    for col, M, label, is_log in (
            (0, Q,  u'$Q$', True),
            (1, DF, u'$\\Delta\\nu_\\mathrm{res}$ / %s' % unit, False)):
        cax = fig.add_subplot(gs[0, col])
        ax  = fig.add_subplot(gs[1, col])
        axc = fig.add_subplot(gs[2, col])
        D = np.ma.masked_invalid(M)

        if is_log:
            pos = M[np.isfinite(M) & (M > 0)]
            norm = mcolors.LogNorm(vmin=pos.min(), vmax=pos.max()) if pos.size else None
        else:
            lim = np.nanmax(np.abs(M))
            norm = mcolors.Normalize(vmin=-lim, vmax=lim)

        im = ax.pcolormesh(xe, ye, D, norm=norm, cmap='viridis')
        set_bg(ax, 'black')
        ax.set_xlim(-180, 180); ax.set_ylim(ye[0], ye[-1])
        ax.set_xticks([-180, -90, 0, 90, 180]); ax.set_xticklabels([])
        ax.set_ylabel(u'Fb. gain / dB')

        cb = fig.colorbar(im, cax=cax, orientation='horizontal')
        cax.xaxis.set_ticks_position('top')
        cax.xaxis.set_label_position('top')
        # FEW ticks, explicitly. Matplotlib's default count is chosen for a tall
        # vertical colourbar; on a horizontal one it packs in far too many.
        if is_log:
            cb.locator = LogLocator(base=10.0, numticks=4)
        else:
            cb.locator = MaxNLocator(nbins=5, symmetric=True, prune=None)
        cb.update_ticks()
        cax.tick_params(labelsize=9, pad=2)
        cax.set_xlabel(label, labelpad=7)

        if is_log and MARK_COMPRESSED:
            ii, jj = np.where(np.isfinite(M) & ~np.isfinite(LIN))
            if len(ii):
                ax.plot(x[jj], GAINS_DB[ii], 'o', mfc='none', mec='w',
                        ms=4, mew=0.9, linestyle='none')
                ax.set_xlim(-180, 180); ax.set_ylim(ye[0], ye[-1])

        for gdb, colour in zip(CUT_GAINS_DB, CUT_COLOURS):
            i = int(np.argmin(np.abs(GAINS_DB - gdb)))
            ax.axhline(GAINS_DB[i], color=colour, ls='--', lw=1.1, dashes=(5, 3))
            axc.plot(x, M[i], '-', color=colour, lw=1.2)

        axc.set_xlim(-180, 180); axc.set_xticks([-180, -90, 0, 90, 180])
        axc.xaxis.set_major_formatter(deg)
        axc.set_xlabel(u'Feedback phase')
        axc.grid(color='0.75', lw=0.6); axc.set_axisbelow(True)
        if is_log:
            axc.set_yscale('log'); axc.set_ylabel(u'$Q$')
        else:
            axc.set_ylabel(u'$\\Delta\\nu_\\mathrm{res}$ / %s' % unit)
            axc.yaxis.set_major_locator(MaxNLocator(nbins=5, symmetric=True))

    # Panel letters, clear of the colourbar tick labels above the axes.
    fig.text(0.012, 0.962, '(a)', fontsize=12)
    fig.text(0.522, 0.962, '(b)', fontsize=12)
    plt.show()


paper_atlas()
print('black = self-oscillation (or unresolved). White rings = too loud to be a')
print('linear measurement. Both are drawn, neither is fitted.')
print('phase axis is relative to %.0f deg (the narrowing phase, taken from the'
      % PHASE_REF)
print('atlas itself), so 0 on the plots is that phase.')
print('gain cuts at %s dB (snapped to the grid).'
      % ', '.join('%+.1f' % g for g in CUT_GAINS_DB))

# -----------------------------------------------------------------------------
#  T_eff / T0 - not in the paper, so it keeps its own panel rather than being
#  forced into a layout designed for two quantities.
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.2, 4.2))
D = np.ma.masked_invalid(M_T)
pos = M_T[np.isfinite(M_T) & (M_T > 0)]
xT, MT = _phase_axis(M_T)
im = ax.pcolormesh(_edges(xT), _edges(GAINS_DB), np.ma.masked_invalid(MT),
                   norm=(mcolors.LogNorm(vmin=pos.min(), vmax=pos.max())
                         if pos.size else None), cmap='viridis')
set_bg(ax, 'black')
ax.set_xlim(-180, 180); ax.set_xticks([-180, -90, 0, 90, 180])
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, p: u'%d\u00b0' % int(round(v))))
ax.set_xlabel(u'Feedback phase'); ax.set_ylabel(u'Fb. gain / dB')
ax.set_title(u'$T_\\mathrm{eff}/T_0$  (the map the paper never drew)')
cb2 = plt.colorbar(im, ax=ax)
cb2.locator = LogLocator(base=10.0, numticks=5); cb2.update_ticks()
plt.tight_layout(); plt.show()


---
## Step 3 — Do the maps obey the physics, and can they be trusted as a lookup?

Three checks, using data already taken plus three confirmation measurements:

1. **The temperature law (Eq. 8).** `T_eff/T0` must equal `Q_eff/Q0` at every point — a
   straight line of slope 1 through hundreds of independent measurements.
2. **Linearity in gain.** Along the narrowing and broadening phases, `Q0/Q_eff = 1 ∓ K`
   with K proportional to gain. Both columns must give the same K.
3. **Reproducibility.** Three settings taken off the map, re-applied cold, re-measured. If
   the system lands where the map says, the map is a trustworthy lookup table — which is
   exactly what Step 4 assumes.

> **Real coil.** Mostly unchanged, and this is the most valuable stage of the three. All three
> checks compare the data against *itself* rather than against a number you supplied, so they
> stay meaningful with no known answer. The one thing to drop is the open-loop Q assertion in
> Step 0, which has nothing to compare against.


In [ ]:
# --- 1. the temperature law --------------------------------------------------
sel  = LINEAR & np.isfinite(M_T)
comp = STAB & np.isfinite(M_Q) & np.isfinite(M_T) & ~LINEAR
qr, tr = (M_Q / Q_OL)[sel], M_T[sel]
# A median, not a mean: one wild point moves it by one rank, not twenty decades.
slope_T = float(np.median(tr / qr))
scatter = float(np.median(np.abs(tr / (slope_T * qr) - 1.0)))

plt.close('all')
plt.figure(figsize=(6.5, 5))
plt.loglog(qr, tr, '.', ms=3, alpha=0.5, label='linear (fitted): %d' % sel.sum())
if comp.sum():
    plt.loglog((M_Q / Q_OL)[comp], M_T[comp], 'x', ms=4, color='0.55',
               label='compressed (excluded): %d' % comp.sum())
xx = np.array([qr.min(), qr.max()])
plt.loglog(xx, xx, 'r--', lw=1, label='Eq. 8:  T_eff/T0 = Q_eff/Q0')
plt.xlabel('Q_eff / Q0'); plt.ylabel('T_eff / T0 (bump area ratio)')
plt.title('The fluctuation-dissipation check: %d points' % sel.sum())
plt.grid(alpha=0.4); plt.legend(loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()
print('temperature law : median T/Q %.3f (physics: 1.000), scatter %.1f%%, '
      '%d excluded' % (slope_T, 100 * scatter, comp.sum()))
ok_T = abs(slope_T - 1.0) < 0.15

# --- 2. linearity in gain along the two damping phases -----------------------
def _narrowing_phase():
    """The phase that narrows, from a cosine fit to K(phase) on the strongest
    mostly-linear row. A fit uses all 24 columns, so a few dropped points
    cannot move it - unlike the global maximum of the map, which sits hard
    against the boundary exactly where points get dropped."""
    rows = [i for i in range(NG) if LINEAR[i].sum() >= 0.8 * NP]
    i = rows[-1] if rows else int(np.argmax(LINEAR.sum(axis=1)))
    good = LINEAR[i]
    K  = 1.0 - Q_OL / M_Q[i]
    ph = np.radians(PHIS)
    A  = np.column_stack([np.cos(ph), np.sin(ph), np.ones(NP)])[good]
    a, b, _ = np.linalg.lstsq(A, K[good], rcond=-1)[0]
    phi0 = np.degrees(np.arctan2(b, a)) % 360.0
    return int(np.argmin(np.abs((PHIS - phi0 + 180.0) % 360.0 - 180.0))), i, phi0


jn, i_ref, phi_fit = _narrowing_phase()
jb = (jn + NP // 2) % NP
print('narrowing phase %.0f deg (cosine fit says %.1f, from the %+.1f dB row)'
      % (PHIS[jn], phi_fit, GAINS_DB[i_ref]))

Ks = []
for j, sgn in ((jn, +1), (jb, -1)):
    # Q0/Q_eff = 1 - K narrowing, 1 + K broadening, so K = sgn*(1 - Q0/Q_eff)
    # is positive along both columns.
    K_meas = sgn * (1.0 - Q_OL / M_Q[:, j])
    # Near K = 1 this is the small difference of two nearly equal numbers, so a
    # 1% error in Q_eff becomes a large error in K. Fit where it is conditioned.
    good = LINEAR[:, j] & np.isfinite(K_meas) & (np.abs(K_meas) < 0.85)
    G, K = GAINS[good], K_meas[good]
    # Physics fixes the intercept: no feedback, no change. A free intercept lets
    # curvature hide inside an offset.
    k_per_A = float(np.sum(G * K) / np.sum(G * G))
    rms     = float(np.sqrt(np.mean((K - k_per_A * G) ** 2)))
    _, c0   = np.polyfit(G, K, 1)          # free-intercept fit, as a check only
    Ks.append((k_per_A, rms, float(c0), int(good.sum())))
print('K per unit gain : narrowing %.3f, broadening %.3f (should agree)'
      % (Ks[0][0], Ks[1][0]))
print('                  rms %.4f / %.4f on %d / %d points; stray intercepts '
      '%+.4f / %+.4f (want 0)'
      % (Ks[0][1], Ks[1][1], Ks[0][3], Ks[1][3], Ks[0][2], Ks[1][2]))
ok_lin = abs(Ks[0][0] / Ks[1][0] - 1.0) < 0.10
if not ok_lin:
    print()
    print('  The noise baseline is already removed by _lorentz_core, so this is')
    print('  not the floor bias that would otherwise explain it. The next suspect')
    print('  is compression: K on the')
    print('  narrowing column is fitted from points nearer the boundary, where')
    print('  the signal inside the resonator is largest, so it reads low. Check')
    print('  Q_LIN from Step 1 - if it sits inside the fitted range, lower the')
    print('  drive (raise TANK_SH) and re-scan. Do not loosen this gate.')

# --- 3. reproducibility: three settings re-applied cold ----------------------
picks = []
qmap = np.where(LINEAR, M_Q, np.nan)
i_, j_ = np.unravel_index(np.nanargmax(qmap), qmap.shape); picks.append((i_, j_, 'highest usable Q'))
dmap = np.where(LINEAR & np.isfinite(M_DF), np.abs(M_DF), np.nan)
i_, j_ = np.unravel_index(np.nanargmax(dmap), dmap.shape); picks.append((i_, j_, 'largest shift'))
picks.append((NG // 2, jb, 'mid-gain broadening'))

print()
worst = 0.0
for i, j, name in picks:
    ps.set(GAINS[i], PHIS[j]); reso.clear(); time.sleep(0.02)
    fc, lw, pk, area, coh = noise_sweep(reso.f0_q)
    ps.off(); reso.clear()
    if not np.isfinite(lw):
        print('  %-20s UNRESOLVED on re-measurement' % name)
        worst = max(worst, 9.9); continue
    # Compared on a log scale: symmetric in over/under, so a wild fit can never
    # print as a tidy-looking "+100%".
    dq = abs(np.log((fc / lw) / M_Q[i, j]))
    worst = max(worst, dq)
    print('  %-20s Q %.0f (map %.0f, %+.0f%%)   shift %+.0f Hz (map %+.0f)'
          % (name, fc / lw, M_Q[i, j], 100 * (np.exp(dq) - 1.0),
             fc - F_OL, M_DF[i, j]))
ok_rep = worst < np.log(1.15)

print()
print('STEP 3:', 'PASS' if (ok_T and ok_lin and ok_rep) else
      'FAIL - the maps disagree with the physics or with themselves')

---
## Step 4 — The designer: ask for what you want, learn what it costs

The maps are now a validated lookup table, so use them as one:

- `report(gain_db, phase)` — one operating point in full: measured Q, shift and
  temperature, plus what follows from them — ring-up time, detection bandwidth, distance
  from the self-oscillation boundary, and the decimation and record length the acquisition
  should use there.
- `design_Q(target)` — the stable setting nearest a wanted Q, refusing anything within two
  grid steps of the boundary.
- `design_shift(hz)` — the same, for decoupling shifts.

The table at the end gives the four canonical operating modes with their trade-offs in
words: the practical summary of the whole atlas.

> **Real coil.** Unchanged. The designer reads whatever atlas was measured, so it works the
> same way on real data — with the caveat that its stability margins are only as good as the
> boundary the scan actually found.


In [ ]:
def _margin_map():
    """Distance in grid steps to the nearest oscillating point, phase wrapping.
    Computed once. Returns inf everywhere if the scan found none - that is not
    a large margin, it is NO INFORMATION, and a sentinel number here would pass
    every '>= 2 steps clear' test in this cell."""
    M   = np.full((NG, NP), np.inf)
    bad = np.argwhere(~STAB)
    if len(bad) == 0:
        return M
    for i in range(NG):
        for j in range(NP):
            if not STAB[i, j]:
                M[i, j] = 0.0; continue
            di = np.abs(bad[:, 0] - i)
            dj = np.abs(bad[:, 1] - j)
            dj = np.minimum(dj, NP - dj)                 # phase wraps
            M[i, j] = float(np.min(np.maximum(di, dj)))
    return M


MARGIN = _margin_map()
if not np.isfinite(MARGIN).any():
    print('WARNING: no oscillating point on the grid, so no margin can be')
    print('         computed. Gain axis ends at %+.1f dB; the Q2.14 actuator'
          % GAINS_DB[-1])
    print('         ceiling is %+.2f dB. If there is room between those two,'
          % (20 * np.log10(2 - 2 ** -14)))
    print('         raise GAINS_DB; if there is not, lower OSHIFT by a step.')
    print('         See the note beside OSHIFT for why the second is preferred.')
    print()


def _phase_index(phase_deg):
    """Nearest phase column, wrapping symmetrically. Plain modulo lands in
    [0, 360), which makes 350 deg pick the 0 deg column instead of 345."""
    return int(np.argmin(np.abs((PHIS - phase_deg + 180.0) % 360.0 - 180.0)))


def report(gain_db, phase_deg):
    """Everything the atlas knows and implies about one operating point."""
    i = int(np.argmin(np.abs(GAINS_DB - gain_db)))
    j = _phase_index(phase_deg)
    print('operating point: gain %.1f dB, phase %.0f deg  (nearest grid point)'
          % (GAINS_DB[i], PHIS[j]))
    if not STAB[i, j]:
        print('  SELF-OSCILLATING here. Not usable as a resonator.'); return
    if not np.isfinite(M_Q[i, j]):
        print('  stable, but UNRESOLVED at this scan resolution - re-scan with')
        print('  a finer grid before operating here'); return
    if not LINEAR[i, j]:
        print('  stable, but Q = %.0f is past the linear limit of %.0f: the'
              % (M_Q[i, j], Q_LIN))
        print('  signal inside the resonator compresses here, so these numbers')
        print('  are not a measurement. Raise TANK_SH and re-scan.'); return
    q, df, tr = M_Q[i, j], M_DF[i, j], M_T[i, j]
    lw   = (F_OL + df) / q
    tau  = q / (np.pi * F_OL)
    marg = MARGIN[i, j]
    print('  measured : Q = %.0f (x%.2f), shift %+.0f Hz, T_eff/T0 = %.2f'
          % (q, q / Q_OL, df, tr))
    print('  derived  : linewidth %.0f Hz, ring-up tau %.1f us, detection'
          % (lw, 1e6 * tau))
    print('             bandwidth ~ +-%.0f Hz around the centre' % (lw / 2))
    if np.isfinite(marg):
        print('  margin   : %d grid step(s) from self-oscillation %s'
              % (int(marg), '- CLOSE, expect drift sensitivity' if marg < 2 else ''))
    else:
        print('  margin   : UNKNOWN - the scan never found the boundary')
    # Bins should be ~lw/20 so the line is well resolved; spacing = FS/(R*N).
    want_bin = max(lw / 20.0, 1.0)
    for dec in range(8, 17):
        N = FS_CLK / ((1 << dec) * want_bin)
        if N <= 16384:
            break
    N = int(min(max(2 ** int(np.ceil(np.log2(max(N, 256)))), 256), 16384))
    print('  acq plan : R = %d, N = %d  ->  %.1f Hz bins, %.2f s per record,'
          % (1 << dec, N, FS_CLK / ((1 << dec) * N), N * (1 << dec) / FS_CLK))
    print('             ~%d bins across the line'
          % max(1, int(lw / (FS_CLK / ((1 << dec) * N)))))


def _best(mask, cost):
    c = np.where(mask, nan_fill(cost, np.inf), np.inf)
    i, j = np.unravel_index(int(np.argmin(c)), c.shape)
    return (i, j) if np.isfinite(c[i, j]) else None


SAFE = LINEAR & (MARGIN >= 2)      # usable AND two steps clear of the boundary


def design_Q(target):
    """The stable setting nearest a wanted Q, two grid steps clear of danger."""
    hit = _best(SAFE, np.abs(np.log(nan_fill(M_Q, 1e-300) / target)))
    if hit is None:
        print('no safe setting reaches Q = %.0f' % target); return
    print('design_Q(%.0f):' % target); report(GAINS_DB[hit[0]], PHIS[hit[1]])


def design_shift(hz):
    """The stable setting nearest a wanted centre shift (for decoupling)."""
    hit = _best(SAFE & np.isfinite(M_DF), np.abs(M_DF - hz))
    if hit is None:
        print('no safe setting reaches %+.0f Hz' % hz); return
    print('design_shift(%+.0f Hz):' % hz); report(GAINS_DB[hit[0]], PHIS[hit[1]])


print('=' * 74)
print(' THE FOUR OPERATING MODES  (from the measured atlas, not from theory)')
print('=' * 74)
modes = [
    ('HIGH-SNR (loading/tuning)', _best(SAFE, -nan_fill(M_Q, 0.0)),
     'fastest to see anything; hottest - do not trust absolute frequencies here'),
    ('COLD (precision)', _best(SAFE & np.isfinite(M_T), nan_fill(M_T, np.inf)),
     'lowest effective temperature; weakest signal - budget longer averaging'),
    ('DECOUPLED (free evolution)',
     _best(SAFE & np.isfinite(M_DF), -np.abs(nan_fill(M_DF, 0.0))),
     'largest centre shift; the electron evolves unmeasured while parked here'),
    ('NEUTRAL (feedback off)', None,
     'the plant as built: Q = %.0f, the reference for everything above' % Q_OL),
]
for name, hit, note in modes:
    print()
    print('-- %s' % name)
    if hit is not None:
        report(GAINS_DB[hit[0]], PHIS[hit[1]])
    print('   note: %s' % note)
print()
print('STEP 4: designer ready - report(), design_Q(), design_shift() are live')

---
## Step 5 — The self-excited oscillator (paper section IV.D)

Push the gain past the boundary and the resonator's losses go negative: instead of ringing
down it rings *up*, and something must limit it. In the paper the plant is an ion with
second-scale dynamics, so their software loop both sustains and limits the oscillation,
holding a chosen amplitude (their Fig. 12).

Here the plant is thirty thousand times faster — `tau = Q/(pi f0)` is 1.4 µs against a loop
interval of ~45 ms — so the limiting is done by the hardware, gently, and the software only
gets to ask questions. Three of them:

- **5a — where is the boundary?** Bisect the gain until the loop starts feeding itself.
  This also tests a prediction: Step 3 fitted `K = k·A` entirely on the stable side, so
  onset should be at `A = 1/k`.
- **5b — how steady is it?** Sustain it just above onset and measure the frequency scatter,
  the paper's metric, from the slope of the unwrapped phase rather than an FFT bin.
- **5c — could software have placed the amplitude?** With a hard clip the amplitude is set
  by the clip level and gain only chooses between dead and saturated. With a gentle limiter
  the equilibrium moves smoothly and a setpoint exists. Rather than assume, measure: sweep
  the gain a few actuator steps around onset and read the answer off the curve.

> **Real coil.** The bisection uses the same sustain test as Step 1, so it needs the same
> replacement — see the note above Step 1. Everything else transfers: the frequency-stability
> measurement is plant-independent, and the amplitude-versus-gain sweep in 5c is a direct
> measurement that answers itself.
>
> One prediction worth checking early: the soft limiting seen here comes from the emulated
> resonator's 25-bit state running out of headroom, which is an artefact of the emulation. A
> real coil and preamplifier have a different nonlinearity, so the real system may well have a
> boundary that can be crossed properly.


In [ ]:
# --- 5a. where is the boundary? ---------------------------------------------
# The atlas only samples points; the boundary lies between them. Bisection
# walks straight to it, using the same sustain test as the scan - if these two
# ever disagree, Step 5 is not finding the boundary the atlas drew.
PHI_N = PHIS[jn]

reso.noise_tank(TANK_SH); reso.noise_out(None)
gen.amp(0.0)
acq.start(dec_log2=10); acq.tune(F_OL)


def oscillating(A, phi, dwell=0.010):
    ps.set(A, phi); reso.clear(); time.sleep(dwell)
    if reso.clipped():
        ps.off(); reso.clear(); return True
    p_on = float(np.var(capture(F_OL, 10, 512)))
    reso.noise_tank(None)
    p_off = float(np.var(capture(F_OL, 10, 512)))
    reso.noise_tank(TANK_SH); ps.off(); reso.clear()
    return p_off >= SUS_OSC * max(p_on, 1e-300)


A_MAX  = 1.99      # A*16384 = 32604, just inside int16. At A = 2.0 the
                   # coefficient is 32768, which wraps to -32768 and silently
                   # inverts the feedback sign. Do not raise this.
lo, hi = 0.05, A_MAX
A_TH   = np.nan
assert not oscillating(lo, PHI_N), 'already oscillating at the bottom of the axis'
if not oscillating(hi, PHI_N):
    print('no onset below A = %.2f (the int16 ceiling). Either K saturates'
          % A_MAX)
    print('below 1 at this drive level, or the narrowing phase is wrong.')
else:
    for _ in range(20):
        mid = np.sqrt(lo * hi)          # geometric: constant fractional step
        if oscillating(mid, PHI_N):
            hi = mid
        else:
            lo = mid
    A_TH = float(np.sqrt(lo * hi))

# Step 3 fitted K = k*A entirely on the stable side, so onset (K = 1) should be
# at A = 1/k. It will not match exactly: k is fitted from points that are
# already slightly compressed, so it reads low and 1/k reads high. The gap
# measures how much the soft limiter bites before the boundary.
A_TH_PRED = 1.0 / Ks[0][0]
print('self-oscillation onset at phase %.0f deg:' % PHI_N)
print('  measured  A_th = %s' % ('%.4f (%+.2f dB)' % (A_TH, 20 * np.log10(A_TH))
                                 if np.isfinite(A_TH) else 'not found'))
print('  predicted A_th = %.4f  from K = %.3f per unit gain (Step 3)'
      % (A_TH_PRED, Ks[0][0]))
if np.isfinite(A_TH):
    print('  gap %+.0f%% - the linear model %s how much gain this loop takes'
          % (100 * (A_TH / A_TH_PRED - 1.0),
             'over-estimates' if A_TH < A_TH_PRED else 'under-estimates'))
ok_th = bool(np.isfinite(A_TH))

if not ok_th:
    print()
    print('STEP 5: FAIL - no onset located, so there is nothing to sustain,')
    print('        nothing to time, and no amplitude to place.')
else:
    # --- 5b. sustain it, and measure the frequency --------------------------
    def osc_freq(nsamp=8192, dec=10):
        """Frequency of a strong tone from the slope of the unwrapped phase.
        An FFT bin here is f_out/N = 60 Hz, two orders coarser than the
        scatter being measured, so a bin peak would report its own
        resolution. Returns (frequency, rms amplitude, coherence)."""
        z  = capture(F_OL, dec, nsamp)
        ph = np.unwrap(np.angle(z))
        sl = float(np.polyfit(np.arange(len(z), dtype=float), ph, 1)[0])
        return (F_OL + sl * acq.f_out() / (2.0 * np.pi),
                float(np.sqrt(np.var(z))), coherence(z))

    SEO_CYC = 30
    A_RUN   = min(1.05 * A_TH, A_MAX)
    ps.set(A_RUN, PHI_N); reso.clear(); time.sleep(0.05)
    fo, ao, co = [], [], []
    for _ in range(SEO_CYC):
        f_, a_, c_ = osc_freq()
        fo.append(f_); ao.append(a_); co.append(c_)
    ps.off(); reso.clear()
    fo, ao, co = np.array(fo), np.array(ao), np.array(co)
    f_sig = float(np.std(fo))
    a_rel = float(np.std(ao) / np.mean(ao))
    print()
    print('sustained at A = %.4f (1.05 x onset):' % A_RUN)
    print('  tone fraction %.2f, amplitude %.3e rms (%.2f%% scatter)'
          % (tone_fraction(float(np.mean(co))), float(np.mean(ao)), 100 * a_rel))
    print('  frequency  mean %+.0f Hz from f0, sigma %.1f Hz over %d records'
          % (np.mean(fo) - F_OL, f_sig, SEO_CYC))

    # --- 5c. can a software loop place the amplitude? -----------------------
    # With a HARD clip the amplitude is set by the clip level and gain only
    # chooses between dead and saturated - no setpoint to aim at. With a SOFT
    # limiter the equilibrium moves smoothly with gain and a setpoint exists.
    # This loop limits softly, so the question is empirical: measure the
    # equilibrium amplitude against gain in single actuator steps.
    A_LSB = ps.gain_lsb(PHI_N)
    steps = np.array([-4, -2, -1, 0, 1, 2, 4, 8, 16, 32, 64])
    held, hcoh = [], []
    for k in steps:
        ps.set(max(A_TH + k * A_LSB, 0.0), PHI_N); reso.clear(); time.sleep(0.05)
        z = capture(F_OL, 10, 2048)
        held.append(float(np.sqrt(np.var(z)))); hcoh.append(coherence(z))
        ps.off(); reso.clear()
    held, hcoh = np.array(held), np.array(hcoh)
    print()
    print('amplitude vs gain, in actuator steps about onset (1 step = %.2e in A,'
          % A_LSB)
    print('the 14-bit coefficient LSB at this phase):')
    for k, h, c in zip(steps, held, hcoh):
        print('    %+3d LSB   %.4e   tone %.2f%s'
              % (k, h, tone_fraction(c), '   <- onset' if k == 0 else ''))

    run = hcoh < 0.5
    if run.sum() >= 3:
        span = float(held[run].max() / max(held[run].min(), 1e-300))
        print()
        print('  across the %d oscillating settings the amplitude spans x%.2f'
              % (int(run.sum()), span))
        graded = span > 1.5
    else:
        span, graded = 1.0, False
        print()
        print('  too few oscillating settings here to judge')
    print('  -> %s' % ('graded: a setpoint exists, an amplitude servo is worth '
                       'building' if graded else
                       'effectively a step: no setpoint to aim at, and a servo '
                       'will limit-cycle'))

    plt.close('all')          # free the previous figures: little memory to spare
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
    ax[0].plot((fo - F_OL) / 1e3, 'o-', ms=3)
    ax[0].set_xlabel('record'); ax[0].set_ylabel('freq - f0 [kHz]')
    ax[0].set_title('frequency: sigma = %.1f Hz' % f_sig); ax[0].grid(alpha=0.4)
    ax[1].plot(ao, 'o-', ms=3)
    ax[1].set_xlabel('record'); ax[1].set_ylabel('amplitude')
    ax[1].set_title('held amplitude: %.2f%% rms' % (100 * a_rel)); ax[1].grid(alpha=0.4)
    ax[2].semilogy(steps, held, 'o-', ms=5)
    ax[2].set_xlabel('gain steps above onset [LSB]'); ax[2].set_ylabel('amplitude')
    ax[2].set_title('is there a setpoint to aim at?'); ax[2].grid(alpha=0.4, which='both')
    plt.tight_layout(); plt.show()

    print()
    ok5 = ok_th and np.isfinite(f_sig) and a_rel < 0.05
    print('STEP 5:', 'PASS' if ok5 else
          'FAIL - onset not located, or the oscillation is not held')

---
## Stage 10 summary

| Step | What it delivered |
|---|---|
| 1 | The full gain/phase atlas measured in noise, with the self-oscillation boundary located and the trustworthy region of the map measured rather than assumed |
| 2 | The paper's Fig. 8 heatmaps, plus the effective-temperature map the paper never images |
| 3 | Eq. 8 tested at every usable point; K checked on both damping phases against a fit forced through the physical origin; three settings re-applied cold |
| 4 | The designer: `report()`, `design_Q()`, `design_shift()`, and the four canonical modes |
| 5 | Onset located by bisection and checked against Step 3's extrapolation; frequency stability measured; the reachability of amplitude control settled by measurement |

### The baseline fix (Stages 8, 9 and 10)

`1/p` is a parabola in `f` only for a *pure* Lorentzian. Any additive floor is absorbed
into that parabola and inflates the fitted width — negligible where the peak towers over
the floor, severe where it does not, because peak power falls as the square of the Q
ratio. A floor worth 2% of the open-loop peak is worth 32% of a 4× broadened one.

`_lorentz_core` now finds the floor by search: for each trial floor, subtract it, fit, and
rebuild the full Lorentzian-plus-floor; the trial that best reproduces the measured points
wins. A wing estimate was tried first and rejected — it needs points beyond ~1.5
linewidths, which a broad line does not have on this grid, i.e. it fails exactly where it
is needed. Measured effect, after deconvolution:

| `Q_eff/Q0` | Q error before | Q error after | `T_eff` error before | after |
|---|---|---|---|---|
| 0.25 | −60% | −2% | +165% | +2% |
| 0.50 | −20% | −3% | +26% | +4% |
| 1.00 | −10% | −9% | +9% | +8% |
| ≥ 2 | ≤1% | ≤1% | ≤7% | ≤7% |

On clean, high-signal data — Stage 8's driven sweep, Stage 9's spectra — the search
returns a zero floor and the result is bit-identical to the old fit, so those stages keep
their existing numbers.

> **Real coil.** Unchanged, and worth running either way — it leaves the board in the state a
> power cycle would.


In [ ]:
# Board back to power-on behaviour.
reso.noise_tank(None); reso.noise_out(None)
ps.off()
gen.stop()
reso.disable()
acq.w(0x00, 0)
print('all quiet: feedback open, generator stopped, resonator disabled')